[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.7 MB/s eta 0:00:00


In [2]:
import torch
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
  B,seq,dim=Q.shape

  attention=torch.matmul(Q,K.transpose(-1,-2))/math.sqrt(dim)

  # calculate mask

  row=torch.arange(seq).view(-1,1)
  col=torch.arange(seq).view(1,-1)

  distance=torch.abs(row-col)
  mask=distance>window_size

  attention_scores=attention.masked_fill_(mask,float("-inf"))
  attention_weights=torch.softmax(attention_scores,dim=-1)

  return torch.matmul(attention_weights,V)



In [6]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [7]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (18.7ms)
  ✅ [2/5] window_size=0 — only sees itself (0.9ms)
  ✅ [3/5] Large window equals full attention (6.1ms)
  ✅ [4/5] Distant tokens don't affect output (5.6ms)
  ✅ [5/5] Gradient flow (27.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (58.5ms total)
  Progress saved. Run status() to see your dashboard.

